# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import math
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange


In [2]:
def process_input_dataframe(dataframe):
    # 1. Convert data types
    # 2. Remove lines with empty values
    # 3. Shuffle to introduce randomness
    # 4. Normalize values
    dataframe.apply(pd.to_numeric, errors='coerce')
    dataframe.dropna(inplace=True)

    return dataframe.sample(frac=1)


def normalize_array(array, min_value, max_value):
    return (array - min_value) / (max_value - min_value)


def calculate_normalization_boundaries_for_columns(dataframe, target_columns):
    boundaries = {}

    for column_name in target_columns:
        boundaries[column_name] = {
            'min': get_column_min(dataframe, column_name),
            'max': get_column_max(dataframe, column_name)
        }

    return boundaries


def normalize_dataframe(dataframe, normalization_boundaries_dict):
    for column_name, boundaries_dict in normalization_boundaries_dict.items():
        min_value = boundaries_dict['min']
        max_value = boundaries_dict['max']

        column_arr = np.array(dataframe[column_name])

        dataframe[column_name] = normalize_array(column_arr, min_value, max_value)


def get_column_min(dataframe, column_name):
    return dataframe[column_name].min()


def get_column_max(dataframe, column_name):
    return dataframe[column_name].max()

#### Reading and pre-processing data

In [3]:
train_file_path = 'data/wind_turbines_training.csv'
test_file_name_path = 'data/wind_turbines_testing.csv'

data_columns = ['Erosion', 'Corosion', 'Crack', 'Pristine', 'Type']

train_df = pd.read_csv(train_file_path, header=None, names=data_columns)
test_df = pd.read_csv(test_file_name_path, header=None, names=data_columns)

train_df = process_input_dataframe(train_df)
test_df = process_input_dataframe(test_df)

#### Displaying basic information about the initial data


In [4]:
def display_dataframe_info(dataframe):
    print('****** General Information ******')
    display(dataframe.info())
    display(dataframe.describe())

    print('****** Data Sample ******')
    display(dataframe.head())


In [5]:
print('==== Train Data Info ====')
display_dataframe_info(train_df)
print('========================', end='\n\n')

print('==== Test Data Info ====')
display_dataframe_info(test_df)
print('=========================')

==== Train Data Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 160 entries, 75 to 74
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Erosion   160 non-null    float64
 1   Corosion  160 non-null    float64
 2   Crack     160 non-null    float64
 3   Pristine  160 non-null    float64
 4   Type      160 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 7.5 KB


None

,Erosion,Corosion,Crack,Pristine,Type
count,160.000000,160.000000,160.000000,160.000000,160.000000
mean,0.184995,0.189979,0.301697,0.323324,3.050000
std,0.257560,0.115119,0.175197,0.330933,1.050906
min,0.000400,0.003500,0.036800,0.000000,1.000000
25%,0.007875,0.095125,0.146800,0.003500,3.000000
50%,0.057300,0.173550,0.292200,0.211900,3.000000
75%,0.260300,0.266575,0.449700,0.647875,4.000000
max,0.938000,0.450500,0.645100,0.927100,4.000000


****** Data Sample ******


,Erosion,Corosion,Crack,Pristine,Type
75,0.3861,0.1567,0.4526,0.0046,3
13,0.0009,0.0568,0.0550,0.8872,4
118,0.5341,0.0742,0.3911,0.0007,1
129,0.0065,0.1608,0.1426,0.6901,4
151,0.0071,0.1579,0.1769,0.6580,4



==== Test Data Info ====
****** General Information ******
<class 'pandas.core.frame.DataFrame'>
Index: 64 entries, 43 to 48
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Erosion   64 non-null     float64
 1   Corosion  64 non-null     float64
 2   Crack     64 non-null     float64
 3   Pristine  64 non-null     float64
 4   Type      64 non-null     int64  
dtypes: float64(4), int64(1)
memory usage: 3.0 KB


None

,Erosion,Corosion,Crack,Pristine,Type
count,64.000000,64.000000,64.000000,64.000000,64.000000
mean,0.144839,0.185519,0.300247,0.369386,3.250000
std,0.227542,0.110623,0.194806,0.357955,0.959497
min,0.000400,0.013400,0.036800,0.000300,1.000000
25%,0.005100,0.095125,0.138325,0.003850,3.000000
50%,0.038800,0.163950,0.251200,0.294500,3.500000
75%,0.165575,0.266250,0.499850,0.704675,4.000000
max,0.878100,0.417200,0.618500,0.927100,4.000000


****** Data Sample ******


,Erosion,Corosion,Crack,Pristine,Type
43,0.54699,0.1297,0.3075,0.0150,1
31,0.00050,0.0387,0.0412,0.9196,4
33,0.00230,0.0934,0.1011,0.8031,4
13,0.34270,0.1579,0.4965,0.0030,3
11,0.32040,0.1440,0.5337,0.0019,3


#### Determining normalization boundaries and performing normalization fot both the training and testing sets


In [6]:
columns_to_be_normalized = ['Erosion', 'Corosion', 'Crack', 'Pristine']
normalization_boundaries_dict = calculate_normalization_boundaries_for_columns(train_df, columns_to_be_normalized)

print('====== Data Normalization ======', end='\n')

print('Based on the input training data, each column will be normalized using the following boundaries:', end='\n')

for column_name, boundaries_dict in normalization_boundaries_dict.items():
    min_value = boundaries_dict['min']
    max_value = boundaries_dict['max']

    print(f'{column_name}: [{min_value}; {max_value}]', end='\n')

normalize_dataframe(train_df, normalization_boundaries_dict)
normalize_dataframe(test_df, normalization_boundaries_dict)

print('\nData has been normalized')
print('================================', end='\n')

====== Data Normalization ======
Based on the input training data, each column will be normalized using the following boundaries:
Erosion: [0.0004; 0.938]
Corosion: [0.0035; 0.4505]
Crack: [0.0368; 0.6451]
Pristine: [0.0; 0.9271]

Data has been normalized


#### Declaring Utils for Layers


In [7]:
def kronecker_delta(i, j):
    return 1 if i == j else 0

#### Declaring Layers


Fuzzifying Layer

In [30]:
class FuzzifyingLayer:
    def __init__(self, membership_functions_count, input_vector_length, c_initial):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length
        self.c = []
        self.s = (membership_functions_count - 1) ** -1

        for i in range(self.input_vector_length):
            self.c.append(c_initial)

    def fuzzify(self, input_vector):
        memberships_matrix = []

        for i in range(self.input_vector_length):
            x = input_vector[i]
            x_memberships = []

            for j in range(self.membership_functions_count):
                membership = self.calculate_membership_for_x(x, i, j)
                x_memberships.append(membership)

            memberships_matrix.append(x_memberships)

        # vector_memberships = [
        #     x1: [ mu1(x1), mu2(x1), ..., mu(x1)]
        #     x1: [ mu1(x1), mu2(x1), ..., mu(x1)]
        #     ...
        #     x: [ mu1(x), mu2(x), ..., mu(x)]
        # ]

        # print(f"memberships_matrix: {memberships_matrix}", end='\n\n')
        # print(f"s: {self.c}", end='\n\n')
        # print(f"c: {self.s}", end='\n\n')

        return memberships_matrix

    def calculate_membership_for_x(self, x, i, j):
        c, s = self.c[i][j], self.s

        # membership = math.exp(- (((x - c) ** 2) / (2 * (s ** 2))))
        #
        membership = 1 / (1 + (((x - c) / s) ** 2))


        return membership

    def train(self, v_c, v_s, predicted_value, actual_values, weights, input_vector):
        c_copy = np.empty_like(self.c)

        for r in range(len(self.c)):
            for l in range(len(self.c[r])):
                c_copy[r][l] = self.calculate_new_c(r, l, weights, v_c, predicted_value, actual_values, input_vector)

        self.c = c_copy

    def calculate_new_c(self, r, j, weights, v_c, predicted_value, actual_value, input_vector):
        memberships_matrix = self.fuzzify(input_vector)

        return self.c[r][j] - v_c * self.calculate_de_to_dc(r, j, weights, predicted_value, actual_value, input_vector,
                                                            memberships_matrix)

    def calculate_de_to_dc(self, r, j, weights, predicted_value, actual_value, input_vector, memberships_matrix):
        error_derivative = np.sum(predicted_value - actual_value)

        weighted_outputs = 0

        for l in range(len(weights)):
            weighted_outputs += weights[l] * self.calculate_dy_to_dc(l, r, j, input_vector, memberships_matrix)

        return error_derivative * weighted_outputs

    def calculate_dy_to_dc(self, l, r, j, input_vector, memberships_matrix):
        m = self.calculate_m(memberships_matrix)
        t = self.calculate_t(l, memberships_matrix)

        first_multiplier = (kronecker_delta(l, r) * m - t) / (m ** 2)
        second_multiplier = self.calculate_t_excluding(r, j, memberships_matrix)
        third_multiplier = self.calculate_dm_to_dc(r, j, input_vector)

        dy_to_dc = first_multiplier * second_multiplier * third_multiplier

        return dy_to_dc

    def calculate_dm_to_dc(self, r, j, input_vector):
        x = input_vector[r]
        c = self.c[r][j]
        s = self.s

        return (math.exp(- (((x - c) ** 2) / 2 * (s ** 2))) * (x - c)) / (self.s ** 2)

        # return math.exp(-(((x - c) ** 2) / (2 * (s ** 2)))) * ((x - c) / (s ** 2))

        # return math.exp(-(((x - c) ** 2) / (2 * (s ** 2)))) * ((x - c) / (s ** 2))

    def calculate_m(self, memberships_matrix):
        result = 0

        for p in range(self.membership_functions_count):
            result += self.calculate_t(p, memberships_matrix)

        return result

    def calculate_t(self, l, memberships_matrix):
        result = 1

        for i in range(self.input_vector_length):
            result *= memberships_matrix[i][l]

        return result

    def calculate_t_excluding(self, r, j, memberships_matrix):
        result = 1

        for i in range(self.input_vector_length):
            if i != j:
                result *= memberships_matrix[i][r]

        return result

Aggregation Layer


In [31]:
class AggregatingLayer:
    def __init__(self, membership_functions_count, input_vector_length):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length

    def aggregate(self, memberships_matrix):
        aggregated_memberships = []

        for i in range(self.membership_functions_count):
            product = 1

            for j in range(self.input_vector_length):
                product *= memberships_matrix[j][i]

            aggregated_memberships.append(product)

        return aggregated_memberships

Linear Layer

In [32]:
class LinearLayer:
    def __init__(self, membership_functions_count, input_vector_length):
        self.membership_functions_count = membership_functions_count
        self.input_vector_length = input_vector_length
        self.weights = np.random.uniform(low=-1, high=1, size=self.membership_functions_count).tolist()

    def process(self, aggregated_memberships):
        weighted_aggregated_memberships_sum = 0
        aggregated_memberships_sum = 0

        for i in range(self.membership_functions_count):
            weighted_aggregated_memberships_sum += self.weights[i] * aggregated_memberships[i]
            aggregated_memberships_sum += aggregated_memberships[i]

        return {
            'f1': weighted_aggregated_memberships_sum,
            'f2': aggregated_memberships_sum,
        }

    def train(self, input_data, expected_output_data, plot_weights=False):
        p_vs = []

        for input_vector in input_data:
            p_v = self.calculate_p_vs(input_vector)
            p_vs.append(p_v)

        pseudo_inverse = np.linalg.pinv(p_vs)
        new_weights = np.matmul(pseudo_inverse, expected_output_data)

        if plot_weights:
            plt.clf()
            plt.plot(self.weights, color='r', label='Previous Weights')
            plt.plot(new_weights, color='g', label='New Weights')
            plt.legend()
            plt.show()

        self.weights = new_weights

    def calculate_p_vs(self, memberships_matrix):
        values = []
        p_vs = []

        for i in range(self.membership_functions_count):
            value = 1

            for j in range(self.input_vector_length):
                value *= memberships_matrix[j][i]

            values.append(value)

        for i in range(len(values)):
            numerator = values[i]
            denominator = 0

            for j in range(len(values)):
                denominator += values[j]

            p_vs.append(numerator / denominator)

        return p_vs

Activation Layer

In [33]:
class ActivationLayer:
    def diffuzify(self, fs_dict):
        # return np.round(fs_dict['f1'] / fs_dict['f2'], 0)
        return fs_dict['f1'] / fs_dict['f2']

### WangMendel Fuzzy Neural Network



In [39]:
class WangMendelFuzzyNeuralNetwork:
    def __init__(self, membership_functions_count, input_vector_length, c_initial):
        self.fuzzifying_layer = FuzzifyingLayer(membership_functions_count, input_vector_length, c_initial)
        self.aggregating_layer = AggregatingLayer(membership_functions_count, input_vector_length)
        self.linear_layer = LinearLayer(membership_functions_count, input_vector_length)
        self.activation_layer = ActivationLayer()

    def predict_exact(self, input_vector):
        return np.round(self.predict(input_vector), 0)

    def predict(self, input_vector):
        fuzzifying_layer_output = self.fuzzifying_layer.fuzzify(input_vector)

        # display(fuzzifying_layer_output)
        # plt.matshow(fuzzifying_layer_output)

        aggregating_layer_output = self.aggregating_layer.aggregate(fuzzifying_layer_output)

        # display(aggregating_layer_output)

        linear_layer_output = self.linear_layer.process(aggregating_layer_output)
        activation_layer_output = self.activation_layer.diffuzify(linear_layer_output)

        return activation_layer_output

    def predict_for_matrix(self, input_data):
        predictions = []

        for input_vector in input_data:
            prediction = self.predict(input_vector)
            predictions.append(prediction)

        return predictions

    def predict_and_validate(self, input_data, expected_output_data, detailed_output=False):
        predicted_output_data = self.predict_for_matrix(input_data)

        total_predictions = len(predicted_output_data)
        correct_predictions = 0

        for i in range(total_predictions):
            predicted_value = predicted_output_data[i]
            predicted_value_rounded = np.round(predicted_value, 0)
            expected_value = expected_output_data[i]

            is_predicted_equal_to_expected = predicted_value_rounded == expected_value

            if is_predicted_equal_to_expected:
                correct_predictions += 1

            if detailed_output:
                print(
                    f'Predicted: {predicted_value}; Actual: {expected_value}; {'✅' if is_predicted_equal_to_expected else '❌'}')

        mse = (np.square(predicted_output_data - expected_output_data)).mean()
        accuracy = np.round(correct_predictions / total_predictions * 100, 0)

        print(f'MSE: {mse}')
        print(f'Accuracy: {accuracy}')

        return [mse, accuracy]

    def train(self, epochs, training_data_df, c_nu=0.5, s_nu=0.5, max_epochs_with_no_mse_change=2):
        print(f'Started training for {epochs} epochs', end='\n')

        input_data = np.array(training_data_df.iloc[:, :-1].values, dtype=float)
        expected_output_data = np.array(training_data_df.iloc[:, -1].values, dtype=float)

        start_time = time.time()

        current_mse = 1
        epochs_without_mse_change = 0
        completed_epochs = 0

        training_vectors_count = len(input_data)

        self.train_linear_layer(input_data, expected_output_data)

        for i in trange(epochs, desc='Epochs progress'):
            print(f'Epoch {i + 1} started', end='\n')

            # print(f"====\nCs: {self.fuzzifying_layer.c}\n")
            # print(f"Ss: {self.fuzzifying_layer.s}\n\n====\n")

            # self.train_linear_layer(input_data, expected_output_data)

            for j in trange(20, desc='Data passes per epoch progress', leave=False):
                predicted_data = self.predict_for_matrix(input_data)

                for k in range(training_vectors_count):
                    self.fuzzifying_layer.train(
                        c_nu,
                        s_nu,
                        predicted_data,
                        expected_output_data,
                        self.linear_layer.weights,
                        input_data[k]
                    )

            [mse, accuracy] = self.predict_and_validate(input_data, expected_output_data, detailed_output=True)

            previous_mse = current_mse
            current_mse = mse
            mse_change = previous_mse - current_mse

            if mse_change == 0:
                epochs_without_mse_change += 1

            if epochs_without_mse_change == max_epochs_with_no_mse_change:
                print(f'MSE hasn\'t changed during last {max_epochs_with_no_mse_change} epochs; Training ended',
                      end='\n')
                break

            completed_epochs += 1
            print(f'Epoch {i + 1} completed', end='\n')

        end_time = time.time()
        elapsed_time = np.round(end_time - start_time, 0)

        print(f'Finished training in {elapsed_time}s in {completed_epochs} epochs', end='\n')

    def train_linear_layer(self, input_data, expected_output_data):
        input_data_fuzzified = []

        for input_vector in input_data:
            fuzzified_vector = self.fuzzifying_layer.fuzzify(input_vector)
            input_data_fuzzified.append(fuzzified_vector)

        self.linear_layer.train(input_data_fuzzified, expected_output_data)

    def save_to_file(self, file_name):
        c = self.fuzzifying_layer.c
        s = self.fuzzifying_layer.s
        weights = self.linear_layer.weights

        np.savez(f"{file_name}.npz", c=c, s=s, weights=weights)

    def initialize_from_file(self, file_name):
        file = np.load(f"{file_name}.npz")

        self.fuzzifying_layer.c = file['c']
        self.fuzzifying_layer.s = file['s']
        self.linear_layer.weights = file['weights']


#### Training

In [41]:
class_column_name = 'Type'

min_class = get_column_min(train_df, class_column_name)
max_class = get_column_max(train_df, class_column_name)

membership_functions_count = 30

# TODO: add utility method to extract x_vector and y
input_vector_length = len(np.array(train_df.iloc[:, :-1].values, dtype=float)[0])

c_initial = np.linspace(min_class, max_class, membership_functions_count).tolist()

wangMendelFuzzyNeuralNetwork = WangMendelFuzzyNeuralNetwork(membership_functions_count, input_vector_length, c_initial)

wangMendelFuzzyNeuralNetwork.train(10, train_df, 0.5, 0.5, 2)

Started training for 10 epochs


Epochs progress:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


Data passes per epoch progress:   0%|          | 0/20 [00:00<?, ?it/s]

Predicted: 2276814271.990627; Actual: 3.0; ❌
Predicted: 2277006444.9497166; Actual: 4.0; ❌
Predicted: 2276872584.918388; Actual: 1.0; ❌
Predicted: 2276889773.1390543; Actual: 4.0; ❌
Predicted: 2276893507.5385733; Actual: 4.0; ❌
Predicted: 2276861191.0184298; Actual: 1.0; ❌
Predicted: 2276919633.9378867; Actual: 4.0; ❌
Predicted: 2276804504.937427; Actual: 3.0; ❌
Predicted: 2276695162.60024; Actual: 3.0; ❌
Predicted: 2276770673.9818544; Actual: 4.0; ❌
Predicted: 2276997983.647016; Actual: 4.0; ❌
Predicted: 2276901246.490091; Actual: 4.0; ❌
Predicted: 2276963091.9493155; Actual: 4.0; ❌
Predicted: 2276853194.027081; Actual: 3.0; ❌
Predicted: 2276911873.74488; Actual: 4.0; ❌
Predicted: 2276859586.1317863; Actual: 1.0; ❌
Predicted: 2277012996.330946; Actual: 4.0; ❌
Predicted: 2276922239.396949; Actual: 4.0; ❌
Predicted: 2276792608.961846; Actual: 3.0; ❌
Predicted: 2276778381.097857; Actual: 4.0; ❌
Predicted: 2276973929.4473076; Actual: 4.0; ❌
Predicted: 2276984285.646661; Actual: 4.0; ❌
Pre

Data passes per epoch progress:   0%|          | 0/20 [00:00<?, ?it/s]

Predicted: 2276814271.990627; Actual: 3.0; ❌
Predicted: 2277006444.9497166; Actual: 4.0; ❌
Predicted: 2276872584.918388; Actual: 1.0; ❌
Predicted: 2276889773.1390543; Actual: 4.0; ❌
Predicted: 2276893507.5385733; Actual: 4.0; ❌
Predicted: 2276861191.0184298; Actual: 1.0; ❌
Predicted: 2276919633.9378867; Actual: 4.0; ❌
Predicted: 2276804504.937427; Actual: 3.0; ❌
Predicted: 2276695162.60024; Actual: 3.0; ❌
Predicted: 2276770673.9818544; Actual: 4.0; ❌
Predicted: 2276997983.647016; Actual: 4.0; ❌
Predicted: 2276901246.490091; Actual: 4.0; ❌
Predicted: 2276963091.9493155; Actual: 4.0; ❌
Predicted: 2276853194.027081; Actual: 3.0; ❌
Predicted: 2276911873.74488; Actual: 4.0; ❌
Predicted: 2276859586.1317863; Actual: 1.0; ❌
Predicted: 2277012996.330946; Actual: 4.0; ❌
Predicted: 2276922239.396949; Actual: 4.0; ❌
Predicted: 2276792608.961846; Actual: 3.0; ❌
Predicted: 2276778381.097857; Actual: 4.0; ❌
Predicted: 2276973929.4473076; Actual: 4.0; ❌
Predicted: 2276984285.646661; Actual: 4.0; ❌
Pre

Data passes per epoch progress:   0%|          | 0/20 [00:00<?, ?it/s]

Predicted: 2276814271.990627; Actual: 3.0; ❌
Predicted: 2277006444.9497166; Actual: 4.0; ❌
Predicted: 2276872584.918388; Actual: 1.0; ❌
Predicted: 2276889773.1390543; Actual: 4.0; ❌
Predicted: 2276893507.5385733; Actual: 4.0; ❌
Predicted: 2276861191.0184298; Actual: 1.0; ❌
Predicted: 2276919633.9378867; Actual: 4.0; ❌
Predicted: 2276804504.937427; Actual: 3.0; ❌
Predicted: 2276695162.60024; Actual: 3.0; ❌
Predicted: 2276770673.9818544; Actual: 4.0; ❌
Predicted: 2276997983.647016; Actual: 4.0; ❌
Predicted: 2276901246.490091; Actual: 4.0; ❌
Predicted: 2276963091.9493155; Actual: 4.0; ❌
Predicted: 2276853194.027081; Actual: 3.0; ❌
Predicted: 2276911873.74488; Actual: 4.0; ❌
Predicted: 2276859586.1317863; Actual: 1.0; ❌
Predicted: 2277012996.330946; Actual: 4.0; ❌
Predicted: 2276922239.396949; Actual: 4.0; ❌
Predicted: 2276792608.961846; Actual: 3.0; ❌
Predicted: 2276778381.097857; Actual: 4.0; ❌
Predicted: 2276973929.4473076; Actual: 4.0; ❌
Predicted: 2276984285.646661; Actual: 4.0; ❌
Pre

In [37]:
wangMendelFuzzyNeuralNetwork.save_to_file('models/modified/model_86')

#### Testing

In [1]:
input = [0.0045, 0.167, 0.1653, 0.6632]

columns = ['Erosion', 'Corosion', 'Crack', 'Pristine']
prediction_df = pd.DataFrame(data=[input], columns=columns)

normalize_dataframe(prediction_df, normalization_boundaries_dict)
input_vector = np.array(prediction_df.iloc[0].values, dtype=float)

predictions = wangMendelFuzzyNeuralNetwork.predict_for_matrix(input_vector)


NameError: name 'pd' is not defined

In [37]:
# wangMendelFuzzyNeuralNetwork.predict([0.1525, 0.3146, 0.4696, 0.0633])

class_column_name = 'Type'

min_class = get_column_min(train_df, class_column_name)
max_class = get_column_max(train_df, class_column_name)

membership_functions_count = 30

c_initial = np.linspace(min_class, max_class, membership_functions_count).tolist()

wangMendelFuzzyNeuralNetwork1 = WangMendelFuzzyNeuralNetwork(membership_functions_count, 4, c_initial)
wangMendelFuzzyNeuralNetwork1.initialize_from_file('models/modified/model_86')

input_data = np.array(test_df.iloc[:, :-1].values, dtype=float)
expected_output_data = np.array(test_df.iloc[:, -1].values, dtype=float)

# wangMendelFuzzyNeuralNetwork1.predict_and_validate(input_data, expected_output_data, detailed_output=True)



wangMendelFuzzyNeuralNetwork1.predict(input_data[0])


[1.4032719063629245e-68,
 4.871621005661288e-63,
 1.3152752718757095e-82,
 2.157063041703067e-71,
 9.618263495736933e-53,
 3.9706706496619086e-53,
 1.8185681546214314e-53,
 9.039955770391576e-54,
 4.803102267952011e-54,
 2.6974018103464517e-54,
 1.5877401376595115e-54,
 9.731684883424812e-55,
 6.17909673256662e-55,
 4.0474065597329244e-55,
 2.7255640858193494e-55,
 1.8816011852331608e-55,
 1.328466731465717e-55,
 9.572871926718594e-56,
 7.02820776557344e-56,
 5.249315205406585e-56,
 3.9833262994725513e-56,
 3.067425813612539e-56,
 2.3946717519735724e-56,
 1.8935220684657382e-56,
 1.5152992364471758e-56,
 1.226359330966608e-56,
 1.0031074450488604e-56,
 8.287697225725765e-57,
 6.912690257308884e-57,
 5.818057511364463e-57]

2.0076033857271147